Imports

In [4]:
import pandas as pd
from top2vec import Top2Vec

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Data

In [5]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


Prepare Data

In [6]:
docs = df["Summary"].tolist()

Prepare Model

In [7]:
model = Top2Vec(docs)

2026-03-01 22:07:49,507 - top2vec - INFO - Pre-processing documents for training
C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-03-01 22:07:49,546 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1218.34it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-01 22:07:51,794 - top2vec - INFO - Creating joint document/word embedding
2026-03-01 22:07:54,696 - top2vec - INFO - Creating lower dimension embedding of documents
20

Analyze Topics

In [9]:
topic_sizes, topic_nums = model.get_topic_sizes()
print(topic_sizes)
print(topic_nums)
topic_words, word_scores, topic_nums = model.get_topics(3)
for words, scores, num in zip(topic_words, word_scores, topic_nums):
    print(num)
    print(f"Words: {words}")

[379  80  61]
[0 1 2]
0
Words: ['banks' 'treasury' 'bank' 'financial' 'securities' 'lending' 'funds'
 'loans' 'fdic' 'loan' 'reserve' 'federal' 'stock' 'market' 'credit'
 'liquidity' 'backed' 'institutions' 'asset' 'capital' 'rate' 'releases'
 'department' 'purchase' 'announces' 'billion' 'board' 'percent'
 'eligible' 'primary' 'facility' 'total' 'states' 'preferred' 'covid'
 'program' 'report' 'it' 'will' 'new' 'up' 'through' 'to' 'also' 'for'
 'at' 'its' 'in' 'term' 'under']
1
Words: ['covid' 'states' 'billion' 'report' 'announces' 'federal' 'eligible'
 'facility' 'releases' 'for' 'department' 'the' 'to' 'reserve' 'with'
 'new' 'percent' 'have' 'and' 'through' 'its' 'that' 'in' 'treasury' 'of'
 'as' 'also' 'act' 'from' 'or' 'program' 'by' 'it' 'capital' 'funds'
 'purchase' 'will' 'an' 'at' 'on' 'under' 'up' 'total' 'rate' 'preferred'
 'institutions' 'is' 'their' 'stock' 'lending']
2
Words: ['treasury' 'federal' 'banks' 'financial' 'securities' 'bank' 'report'
 'reserve' 'fdic' 'annou

Analyze Documents

In [10]:
documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=0, num_docs=10)

Prepare Validation

In [11]:
# Top2Vec
top2vec_topics = [list(words) for words in model.get_topics()[0]]

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.corpora import Dictionary

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Preprocess documents for gensim
stop_words = set(stopwords.words('english'))
processed_docs = []
for doc in docs:
    tokens = word_tokenize(doc.lower())
    tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    processed_docs.append(tokens)

# Create dictionary
id2word = Dictionary(processed_docs)

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

def calculate_metrics(topics, corpus_docs, dictionary):
    # 1. Coherence (Cv)
    cm = CoherenceModel(topics=topics, texts=corpus_docs, dictionary=dictionary, coherence='c_v')
    coherence = cm.get_coherence()
    
    # 2. Topic Diversity
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words) if len(all_words) > 0 else 0
    
    return coherence, diversity

# Calculate metrics for Top2Vec
cv_top2vec, div_top2vec = calculate_metrics(top2vec_topics, processed_docs, id2word)
print(f"Top2Vec Coherence (C_v): {cv_top2vec:.4f}")
print(f"Top2Vec Topic Diversity: {div_top2vec:.4f}")

NameError: name 'processed_docs' is not defined